# Hipotezy H1 i H2

Wyniki testów zapisanych w preregistracji [`docs/hipotezy.md`](../docs/hipotezy.md). Liczy je `make hypotheses CONFIRMATORY=1` (`scripts/09_test_hypotheses.py`); ten notatnik tylko je pokazuje. Przebieg potwierdzający ma pierwszeństwo, a eksploracyjny jest wyraźnie oznaczony.

- **H1** (Reddit 2005–2018): maleje zaufanie do konsensusu naukowego. H1-A: udział postów `challenges` wśród postów o nauce (`stance_v1`). H1-B: udział źródeł konsensusu i źródeł alternatywnych wśród postów z powołaniem (`gliner_v1`).
- **H2** (Wikipedia 2020–2026): użytkownicy częściej odwołują się do źródeł alternatywnych; udział linków do blogów, mediów społecznościowych i serwisów wideo wśród linków dodanych do przypisów.

In [1]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from digital_media_analysis.paths import AGREEMENT_DIR, HYPOTHESES_DIR, VALIDATION_DIR

mode = next((name for name in ["confirmatory", "exploratory"] if (HYPOTHESES_DIR / f"{name}_run.json").exists()), None)
if mode is None:
    print("Brak wyników: hipotezy nie zostały jeszcze przetestowane (make hypotheses CONFIRMATORY=1 po commicie preregistracji).")
else:
    run = json.loads((HYPOTHESES_DIR / f"{mode}_run.json").read_text(encoding="utf-8"))
    print(f"Przebieg: {mode}, commit {run['commit'][:7]}, {run['created_utc']}")
    for key, value in run["notes"].items():
        print(f"{key}: {value}")

Brak wyników: hipotezy nie zostały jeszcze przetestowane (make hypotheses CONFIRMATORY=1 po commicie preregistracji).


## Trendy

Iloraz szans na rok z 95% przedziałem i jednostronną wartością p w kierunku zapisanym w preregistracji. Analiza `primary` to test główny; pozostałe wiersze to analizy wrażliwości.

In [2]:
if mode is not None and (HYPOTHESES_DIR / f"{mode}_trends.csv").exists():
    trends = pd.read_csv(HYPOTHESES_DIR / f"{mode}_trends.csv")
    columns = ["hypothesis", "analysis", "outcome", "direction", "odds_ratio_per_year", "ci_low", "ci_high", "p_one_sided", "supported", "units", "positives", "note"]
    display(trends[columns].round(4))

## Udziały w kolejnych latach

Ważone udziały z 95% przedziałami bootstrap. Dla H1-B kolumna `corrected` to udział skorygowany o błąd instrumentu (Rogan–Gladen); przy małej liczbie przypadków w gold jest niepewny.

In [3]:
if mode is not None and (HYPOTHESES_DIR / f"{mode}_yearly.csv").exists():
    yearly = pd.read_csv(HYPOTHESES_DIR / f"{mode}_yearly.csv")
    overall = yearly[yearly["group"].eq("all")]
    panels = list(overall.groupby(["hypothesis", "outcome"]))
    figure, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.4), squeeze=False)
    for axis, ((hypothesis, outcome), rows) in zip(axes[0], panels):
        axis.fill_between(rows["year"], rows["low"], rows["high"], alpha=0.2)
        axis.plot(rows["year"], rows["share"], marker="o")
        if "corrected" in rows and rows["corrected"].notna().any():
            axis.plot(rows["year"], rows["corrected"], linestyle="--", label="skorygowany")
            axis.legend(fontsize=8)
        axis.set_title(f"{hypothesis}: {outcome}", fontsize=10)
        axis.set_ylim(0, 1)
    figure.tight_layout()
    plt.show()
    by_group = yearly[yearly["group"].ne("all")]
    if len(by_group):
        display(by_group.pivot_table(index=["hypothesis", "outcome", "group"], columns="year", values="share").round(2))

## Stanowisko: zgodność annotatorów i walidacja instrumentu

Warunek trafności H1-A: `stance_v1` na zbiorze testowym musi osiągnąć F1 ≥ 0,5 dla `challenges` i κ ≥ 0,4.

In [4]:
for path in [AGREEMENT_DIR / "pilot_stance_agreement.csv", VALIDATION_DIR / "stance_test_metrics.csv", VALIDATION_DIR / "stance_dev_metrics.csv"]:
    if path.exists():
        print(path.name)
        display(pd.read_csv(path).round(3))
    else:
        print(f"{path.name}: jeszcze nie ma (etykiety stanowiska nie zostały zaimportowane albo instrument nie był walidowany).")

pilot_stance_agreement.csv: jeszcze nie ma (etykiety stanowiska nie zostały zaimportowane albo instrument nie był walidowany).
stance_test_metrics.csv: jeszcze nie ma (etykiety stanowiska nie zostały zaimportowane albo instrument nie był walidowany).
stance_dev_metrics.csv: jeszcze nie ma (etykiety stanowiska nie zostały zaimportowane albo instrument nie był walidowany).
